In [ ]:
!pip install torch torchvision transformers datasets -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print(torch.__version__, "GPU:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.1 MB/s eta 0:00:00
2.11.0+cu128 GPU: True


In [ ]:
!pip install torch transformers datasets accelerate evaluate -q

In [ ]:
!pip uninstall -y torchvision -q

In [ ]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
small_train = tokenized["train"].shuffle(seed=42).select(range(2000))
small_test = tokenized["test"].shuffle(seed=42).select(range(500))

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./bert_sentiment",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_steps=50,
    report_to="none",
)

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    acc = (preds == pred.label_ids).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model, args=args,
    train_dataset=small_train, eval_dataset=small_test,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.368554,0.296145,0.874000
2,0.192173,0.451762,0.888000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.192173,0.451762,2,0.888000


{'eval_loss': 0.45176151394844055, 'eval_accuracy': 0.888}

In [ ]:
text = "This movie was surprisingly good, I loved it."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits
print("Positive" if logits.argmax().item() == 1 else "Negative")

Positive
